In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import plotly.io as pio
import kairo

pio.renderers.default = "notebook_connected"
pd.set_option("display.precision", 3)

LOG = "../../data/thesis/general/running_example/L3.xes"
META = ["case:concept:name", "time:timestamp"]
REASSIGNMENT = pd.Timestamp("2020-07-01 12:00", tz="UTC")

In [2]:
log = kairo.read_log(LOG, resource="org:resource", event_duration="event:duration_min")
print(log["case:concept:name"].nunique(), "cases,", len(log), "events")

4920 cases, 24600 events


In [3]:
features = kairo.compute_features_resource(log, feature_window="1D", features=["res_events", "act_res_shares"])
print(len(features), "feature windows,", features.shape[1] - 1, "feature columns")

366 feature windows, 24 feature columns


In [4]:
days = pd.to_datetime(["2020-06-29", "2020-06-30", "2020-07-02", "2020-07-03"]).tz_localize("UTC")
columns = [c for c in features.columns if c.startswith("res_events")] + ["act_res_shares:a by res_01", "act_res_shares:a by res_03"]
features[features["time:timestamp"].isin(days)].set_index("time:timestamp")[columns].T

time:timestamp,2020-06-29 00:00:00+00:00,2020-06-30 00:00:00+00:00,2020-07-02 00:00:00+00:00,2020-07-03 00:00:00+00:00
res_events:res_01,45.000,40.000,3.000,4.000
res_events:res_02,16.000,16.000,32.000,45.000
res_events:res_03,2.000,3.000,26.000,33.000
res_events:res_04,18.000,18.000,5.000,9.000
act_res_shares:a by res_01,0.938,0.875,0.083,0.053
act_res_shares:a by res_03,0.000,0.062,0.833,0.789


In [5]:
values = features.drop(columns="time:timestamp")
unscaled = kairo.compute_pca(features)
print("largest standard deviation, events per resource:", values.filter(like="res_events").std().max().round(2),
      "shares:", values.filter(like="act_res_shares").std().max().round(2))
print("explained variance % without scaling:", np.round(100 * unscaled.explained_variance_ratio_[:3], 1))
pd.Series(unscaled.components_[0], index=unscaled.feature_names_in_).abs().sort_values(ascending=False).head(6).round(2)

largest standard deviation, events per resource: 17.75 shares: 0.41
explained variance % without scaling: [83.8 14.4  1.1]


res_events:res_01             0.71
res_events:res_02             0.47
res_events:res_03             0.46
res_events:res_04             0.23
act_res_shares:d by res_02    0.02
act_res_shares:a by res_01    0.02
dtype: float64

In [6]:
scaled = kairo.standardize(features, method="zscore")
pca = kairo.compute_pca(scaled)
errors = kairo.compute_pca_reconstruction_errors(scaled, pca)
standardized = scaled.drop(columns="time:timestamp")
print("error without components", round(((standardized - standardized.mean()) ** 2).sum(axis=1).mean(), 3))
pd.DataFrame({
    "explained variance %": 100 * pca.explained_variance_ratio_,
    "cumulative %": 100 * pca.explained_variance_ratio_.cumsum(),
    "reconstruction error": errors.to_numpy(),
}, index=errors.index).round(3)

error without components 23.934


,explained variance %,cumulative %,reconstruction error
components,,,
1,52.881,52.881,11.278
2,5.243,58.124,10.023
3,4.840,62.964,8.864
4,4.651,67.615,7.751
5,4.483,72.098,6.678
6,4.349,76.447,5.637
7,4.279,80.726,4.613
8,4.067,84.793,3.640
9,3.877,88.670,2.712


In [7]:
kairo.plot_pca_variances(pca, cut_component=1).show()
kairo.plot_pca_reconstruction_errors(errors, cut_component=1).show()

In [8]:
loadings = pd.Series(pca.components_[0], index=pca.feature_names_in_)
loadings.sort_values().round(3)

act_res_shares:e by res_04   -0.275
act_res_shares:a by res_01   -0.275
act_res_shares:d by res_01   -0.275
act_res_shares:c by res_02   -0.274
act_res_shares:b by res_01   -0.274
res_events:res_01            -0.261
res_events:res_04            -0.245
act_res_shares:b by res_03   -0.017
act_res_shares:d by res_04   -0.016
act_res_shares:d by res_03   -0.011
act_res_shares:e by res_03   -0.009
act_res_shares:c by res_01    0.003
act_res_shares:e by res_01    0.004
act_res_shares:a by res_02    0.005
act_res_shares:c by res_04    0.007
act_res_shares:a by res_04    0.016
act_res_shares:b by res_04    0.025
res_events:res_02             0.230
res_events:res_03             0.254
act_res_shares:b by res_02    0.273
act_res_shares:a by res_03    0.273
act_res_shares:c by res_03    0.274
act_res_shares:e by res_02    0.275
act_res_shares:d by res_02    0.275
dtype: float64

In [9]:
for i in range(1, 12):
    component = pd.Series(pca.components_[i], index=pca.feature_names_in_)
    print(f"pc_{i + 1}", component.abs().sort_values(ascending=False).head(3).round(2).to_dict())

pc_2 {'act_res_shares:b by res_03': 0.57, 'act_res_shares:a by res_04': 0.5, 'act_res_shares:c by res_04': 0.38}
pc_3 {'act_res_shares:e by res_03': 0.49, 'act_res_shares:b by res_04': 0.44, 'act_res_shares:c by res_04': 0.4}
pc_4 {'act_res_shares:e by res_03': 0.57, 'act_res_shares:d by res_03': 0.51, 'act_res_shares:a by res_02': 0.4}
pc_5 {'act_res_shares:e by res_01': 0.61, 'act_res_shares:d by res_04': 0.48, 'act_res_shares:d by res_03': 0.45}
pc_6 {'act_res_shares:a by res_02': 0.52, 'act_res_shares:d by res_03': 0.43, 'act_res_shares:c by res_01': 0.43}
pc_7 {'act_res_shares:d by res_04': 0.5, 'act_res_shares:b by res_04': 0.45, 'act_res_shares:c by res_01': 0.41}
pc_8 {'act_res_shares:c by res_01': 0.64, 'act_res_shares:a by res_02': 0.43, 'act_res_shares:b by res_04': 0.38}
pc_9 {'act_res_shares:c by res_04': 0.65, 'act_res_shares:a by res_02': 0.53, 'act_res_shares:b by res_04': 0.34}
pc_10 {'act_res_shares:b by res_03': 0.58, 'act_res_shares:b by res_04': 0.38, 'act_res_shar

In [10]:
cut = 1
compressed = kairo.apply_pca(scaled, pca, cut_component=cut)
starts = compressed["time:timestamp"]
period = np.select([starts < REASSIGNMENT.floor("1D"), starts > REASSIGNMENT], ["before", "after"], "drift day")
compressed.groupby(period)["pc_1"].agg(["count", "min", "max"]).round(2)

,count,min,max
after,183,1.95,4.56
before,182,-4.36,-2.63
drift day,1,-0.11,-0.11


In [11]:
periods = pd.cut(compressed["time:timestamp"], [compressed["time:timestamp"].min() - pd.Timedelta("1D"), REASSIGNMENT, compressed["time:timestamp"].max()], labels=["before", "after"])

for k in [2, 3, 4, 5, 6]:
    kmeans = kairo.compute_kmeans(compressed, k=k)
    states = kairo.get_kmeans_states(compressed, kmeans)
    print(f"k = {k}: silhouette {kairo.compute_silhouette(compressed, states):.3f}, "
          f"Calinski-Harabasz {kairo.compute_calinski_harabasz(compressed, states):.1f}, "
          f"centers {np.round(np.sort(kmeans.cluster_centers_.ravel()), 2)}")
    print(pd.crosstab(periods, states["cluster"]).to_string(), "\n")

k = 2: silhouette 0.934, Calinski-Harabasz 24778.6, centers [-3.53  3.53]
cluster           0    1
time:timestamp          
before            0  183
after           183    0 

k = 3: silhouette 0.739, Calinski-Harabasz 19149.1, centers [-3.55  3.09  3.83]
cluster           0    1   2
time:timestamp              
before            0  182   1
after           106    0  77 

k = 4: silhouette 0.734, Calinski-Harabasz 15653.7, centers [-3.55  2.76  3.46  4.02]
cluster          0    1   2   3
time:timestamp                 
before           0  182   0   1
after           60    0  92  31 

k = 5: silhouette 0.719, Calinski-Harabasz 13762.1, centers [-3.55  1.95  3.17  3.67  4.15]
cluster          0    1   2  3   4
time:timestamp                    
before           0  182   0  1   0
after           37    0  74  5  67 

k = 6: silhouette 0.539, Calinski-Harabasz 21822.9, centers [-3.8  -3.26  0.92  3.05  3.58  4.07]
cluster          0   1   2  3   4   5
time:timestamp                       
be

In [12]:
for k in [2, 3, 6]:
    kmeans = kairo.compute_kmeans(compressed, k=k)
    states = pd.concat([compressed, kairo.get_kmeans_states(compressed, kmeans)], axis=1)
    print(f"k = {k}:", len(kairo.get_kmeans_log_trajectory(states)), "visits")

k = 2: 2 visits


k = 3: 96 visits
k = 6: 213 visits


In [13]:
k = 2
kmeans = kairo.compute_kmeans(compressed, k=k)
states = pd.concat([compressed, kairo.get_kmeans_states(compressed, kmeans)], axis=1)

first_visit = states.drop_duplicates("cluster")["cluster"].tolist()
states["state"] = states["cluster"].map({cluster: number for number, cluster in enumerate(first_visit, start=1)})
states[states["time:timestamp"].between("2020-06-29", "2020-07-03")]

,time:timestamp,pc_1,cluster,state
180,2020-06-29 00:00:00+00:00,-4.020,1,1
181,2020-06-30 00:00:00+00:00,-4.000,1,1
182,2020-07-01 00:00:00+00:00,-0.108,1,1
183,2020-07-02 00:00:00+00:00,3.231,0,2
184,2020-07-03 00:00:00+00:00,3.316,0,2


In [14]:
trajectory = kairo.get_kmeans_log_trajectory(states)
trajectory

,state,start,end,duration,feature_windows
0,1,2020-01-01 00:00:00+00:00,2020-07-02 00:00:00+00:00,183 days,183
1,0,2020-07-02 00:00:00+00:00,2021-01-01 00:00:00+00:00,183 days,183


In [15]:
colors = kairo.get_kmeans_state_colors(kmeans)
kairo.plot_kmeans_log_trajectory(trajectory, colors).show()

In [16]:
cluster_states = states[["cluster"]]
print(pd.Series({
    "states": k,
    "silhouette": kairo.compute_silhouette(compressed, cluster_states),
    "Calinski-Harabasz": kairo.compute_calinski_harabasz(compressed, cluster_states),
}).round(3).to_string())
print(kairo.get_kmeans_state_distances(kmeans).round(2))

states                   2.000
silhouette               0.934
Calinski-Harabasz    24778.594
      0     1
0  0.00  7.06
1  7.06  0.00


In [17]:
for start, end in [(None, REASSIGNMENT), (REASSIGNMENT, None)]:
    frequencies = kairo.get_kmeans_state_frequencies(states, start_date=start, end_date=end)
    kairo.plot_kmeans_state_frequencies(frequencies, colors).show()

In [18]:
from thesis_figures import write_log_trajectory, write_state_timeline

names = {str(cluster): f"{number}" for cluster, number in zip(states["cluster"], states["state"])}
write_log_trajectory("log_trajectory_resource", trajectory, {name: f"state{'AB'[int(number) - 1]}" for name, number in names.items()},
                     drift=REASSIGNMENT)
write_state_timeline("states_resource", states["time:timestamp"], states["pc_1"], states["state"],
                     y_range=(-5, 5), y_ticks=[-4, 0, 4], y_label="First component", drift=REASSIGNMENT,
                     centers=sorted(kmeans.cluster_centers_.ravel()))

['../../thesis/figures/tikz/states_resource.tex',
 '../../thesis-de/figures/tikz/states_resource.tex']

In [19]:
distributions = kairo.compute_state_distributions(states, signal_window="1D")
drift_signal = kairo.compute_drift_signal(distributions, divergence="kl", reference="previous")
scores = drift_signal.set_index("signal_window")["drift_score"]
print(len(distributions), "signal windows,", (scores > 0).sum(), "drift score above 0, log(1e9) =", np.log(1e9).round(3))
scores.loc["2020-06-30":"2020-07-03"].round(3)

366 signal windows, 1 drift score above 0, log(1e9) = 20.723


signal_window
2020-06-30 00:00:00+00:00     0.000
2020-07-01 00:00:00+00:00     0.000
2020-07-02 00:00:00+00:00    20.723
2020-07-03 00:00:00+00:00     0.000
Name: drift_score, dtype: float64

In [20]:
kairo.plot_drift_signal(drift_signal, divergence="kl", reference="previous").show()

In [21]:
kairo.compute_drift_signal(distributions, divergence="kl", reference="baseline")["drift_score"].round(3).value_counts()

drift_score
0.693    366
Name: count, dtype: int64

In [22]:
monthly = kairo.compute_state_distributions(states, signal_window="30D")
monthly_signal = kairo.compute_drift_signal(monthly, divergence="kl", reference="recent", lookback=3)
windows = states.groupby(states["time:timestamp"].dt.floor("30D")).size().rename("feature windows")
pd.concat([monthly.round(3), windows, monthly_signal.set_index("signal_window")["drift_score"].round(3)], axis=1)

,0,1,feature windows,drift_score
2019-12-10 00:00:00+00:00,0.000,1.000,8,NaN
2020-01-09 00:00:00+00:00,0.000,1.000,30,NaN
2020-02-08 00:00:00+00:00,0.000,1.000,30,NaN
2020-03-09 00:00:00+00:00,0.000,1.000,30,0.000
2020-04-08 00:00:00+00:00,0.000,1.000,30,0.000
2020-05-08 00:00:00+00:00,0.000,1.000,30,0.000
2020-06-07 00:00:00+00:00,0.167,0.833,30,3.003
2020-07-07 00:00:00+00:00,1.000,0.000,30,2.890
2020-08-06 00:00:00+00:00,1.000,0.000,30,0.944
2020-09-05 00:00:00+00:00,1.000,0.000,30,0.325


In [23]:
kairo.plot_drift_signal(monthly_signal, divergence="kl", reference="recent").show()

In [24]:
from thesis_figures import write_drift_signal

write_drift_signal("drift_signal_resource_1d", drift_signal, "1D", REASSIGNMENT, y_max=24, y_ticks=[0, 10, 20])
write_drift_signal("drift_signal_resource_30d", monthly_signal, "30D", REASSIGNMENT, y_max=3.5, y_ticks=[0, 1, 2, 3])

['../../thesis/figures/tikz/drift_signal_resource_30d.tex',
 '../../thesis-de/figures/tikz/drift_signal_resource_30d.tex']